# WEEK 6 — RAG Optimization + LLM Evaluation + LangSmith (Live Session Master)

**Goal:** take the working RAG bot from Week 5 and make it *reliable* — sharper retrieval, real evaluation
scores instead of eyeballing answers, and full tracing so you can see exactly what your bot is doing.

## The arc of today
1. **Optimize** — fix naive retrieval with Multi-Query Retriever + Contextual Compression.
2. **Evaluate** — score answers with Faithfulness and Relevance instead of "it looks fine to me".
3. **Judge** — use LLM-as-a-Judge to compare two candidate answers.
4. **Trace** — open the black box with LangSmith (`@traceable`, datasets, evaluators, feedback).
5. **Compare** — know LangSmith vs. Langfuse well enough to answer "which have you used?" in an interview.

> **REMEMBER THIS:** A working demo and a production-ready RAG system are not the same thing.
> Today's session is the gap between them.

---


## Before We Start — What You Already Have

Last week, in one live session, you built:

| # | Piece | What it does |
|---|-------|---------------|
| 1 | RAG Pipeline | Load → chunk → embed → retrieve → answer |
| 2 | Chunking | Splitting a PDF into searchable pieces |
| 3 | Embeddings & Store | Text → vectors, saved in a vector DB |
| 4 | Retrieval | Pulling the top matching chunks |

You have a bot that **works**. Today we make it **RELIABLE**.


## Part 0 — Rebuild the Foundation (Week 5 recap)

Everything today plugs into the pipeline you already built, so we rebuild it in a few cells first:
load the 3 insurance PDFs → chunk → embed → store → base retriever → base answer chain.

**Enhance this step**
- confirm the OpenAI key and LangSmith key both load
- turn tracing on now, so every cell below is automatically traced to LangSmith as we go


In [ ]:
# Step 0 -- Environment check + turn on LangSmith tracing for the WHOLE notebook
from pathlib import Path
import os
from dotenv import load_dotenv

env_path = Path.cwd() / '.env'
load_dotenv(env_path, override=True)

# Automatic tracing (Part 3, slide 18): one env var and every LangChain call is traced for free.
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ.setdefault('LANGCHAIN_PROJECT', os.getenv('LANGCHAIN_PROJECT', 'week6-rag-optimization'))
os.environ.setdefault('LANGSMITH_PROJECT', os.environ['LANGCHAIN_PROJECT'])

print('OPENAI_API_KEY loaded:', bool(os.getenv('OPENAI_API_KEY')))
print('LANGSMITH_API_KEY loaded:', bool(os.getenv('LANGSMITH_API_KEY')))
print('LangSmith project:', os.environ['LANGCHAIN_PROJECT'])
print('Tracing enabled:', os.environ['LANGSMITH_TRACING'])


In [ ]:
# Packages used in this notebook
# - python-dotenv, langchain-community, langchain-text-splitters, langchain-openai, langchain-classic
# - chromadb, langsmith
# (optional, Part 4) langfuse -- only needed if you want to run the live Langfuse cells


In [ ]:
# Step 1 -- Load the 3 insurance PDFs (same documents as Week 5)
from langchain_community.document_loaders import PyPDFLoader

pdf_files = sorted(Path.cwd().glob('*.pdf'))
docs = []
for pdf_file in pdf_files:
    docs.extend(PyPDFLoader(str(pdf_file)).load())

print('PDF files:', [p.name for p in pdf_files])
print('Loaded pages:', len(docs))


In [ ]:
# Step 2 -- Chunk the pages
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)

print('Chunks created:', len(chunks))
print(chunks[0].page_content[:400])


In [ ]:
# Step 3 -- Embeddings + vector store + base LLM
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

base_retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('Vector store ready. Base retriever k =', base_retriever.search_kwargs['k'])


In [ ]:
# Step 4 -- A minimal answer chain: retrieve -> stuff context -> ask the LLM
from langchain_core.messages import SystemMessage, HumanMessage

ANSWER_SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions about HDFC Life insurance policies. "
    "Answer ONLY using the provided context. If the context does not contain the answer, say so."
)

def answer_question(question: str, retriever, k: int = 4) -> dict:
    """Retrieve chunks with `retriever`, then generate an answer grounded in them."""
    retrieved_docs = retriever.invoke(question)[:k]
    context = "\n\n".join(d.page_content for d in retrieved_docs)

    messages = [
        SystemMessage(content=ANSWER_SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}"),
    ]
    response = llm.invoke(messages)
    return {"question": question, "answer": response.content, "context": context, "docs": retrieved_docs}

result = answer_question("What is the grace period for premium payment?", base_retriever)
print(result["answer"])


## Part 1 — Why "It Works" Isn't Enough

Your bot answers questions. But does it answer them well, every single time?

> **Why this matters:** half the RAG systems that impress in a demo start missing obvious answers
> the moment a real user phrases things their own way.

### The problem with naive retrieval
1. **One phrasing, one shot** — plain similarity search only searches the words you typed, not the
   words the document actually uses.
2. **Noisy top-K** — the top matches often include half-relevant chunks that just add clutter for
   the LLM to wade through.
3. **No feedback loop** — without a score, you can't tell if yesterday's tweak made retrieval better
   or quietly made it worse (this is what Part 2 fixes).

> **REMEMBER THIS:** Naive RAG isn't broken — it's brittle. A small wording difference between the
> user's question and the document's language can break the whole answer.

### A familiar support-desk problem

| A user says... | The document says... |
|---|---|
| "I paid late" | grace period |
| "My claim got denied" | exclusion clause |
| "Extra covers I bought" | riders / add-ons |

One retriever call, one phrasing. Miss the wording, miss the answer.


In [ ]:
# Feel the problem: same intent, two different phrasings, one plain retriever.
casual_question = "what happens if I'm late on my payment?"
formal_question = "what is the grace period for premium payment?"

for q in (formal_question, casual_question):
    docs_for_q = base_retriever.invoke(q)
    print("=" * 70)
    print("Question:", q)
    print("Top chunk preview:", docs_for_q[0].page_content[:150].replace("\n", " "))


**ASK THE ROOM:** Think of one question your users ask in at least three different ways.
That exact gap is what breaks single-query retrieval.

### Two fixes for sharper retrieval — both slot into your existing retriever, no rebuild required

| | What it fixes |
|---|---|
| **Multi-Query Retriever** | Ask the same question several different ways, search each one, then merge the results. |
| **Contextual Compression** | After retrieving, strip each chunk down to only the sentences that actually answer the question. |

> **WRITE THIS DOWN:** Multi-Query fixes *what you search for*. Contextual Compression fixes
> *what survives after you've found it*. Most real systems use both.


### Multi-Query Retriever, step by step

```
1 question -> LLM writes 3-4 phrasings -> search each phrasing -> merge + de-dupe chunks
```

Example: `"What happens if I pay late?"` becomes `"grace period"`, `"late premium penalty"`,
`"missed due date consequences"`.

> **WRITE THIS DOWN:** Multi-Query = ask it several ways, search all of them, then merge and
> de-duplicate. More recall, same document.
> **Tradeoff:** more LLM + retrieval calls = more cost/latency. Worth it when a wrong answer
> costs you a frustrated customer.


In [ ]:
# Multi-Query Retriever: let an LLM generate alternative phrasings, search each, merge + de-dupe.
import logging
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Turning on this logger lets us SEE the generated phrasings (great for a live demo).
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)
logging.basicConfig()

multi_query_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

mq_docs = multi_query_retriever.invoke(casual_question)
print(f"\nMulti-query retrieved {len(mq_docs)} unique chunks for: {casual_question!r}")
for d in mq_docs[:4]:
    print("-", d.page_content[:120].replace("\n", " "))


In [ ]:
# Side-by-side: baseline vs multi-query on the casual phrasing that trips up plain search.
baseline_docs = base_retriever.invoke(casual_question)
mq_docs = multi_query_retriever.invoke(casual_question)

print("Baseline top chunk:  ", baseline_docs[0].page_content[:150].replace("\n", " "))
print("Multi-query chunks:  ", len(mq_docs), "unique chunks pulled from several phrasings")
print("Contains a 'grace period' chunk:",
      any("grace period" in d.page_content.lower() for d in mq_docs))


### Contextual Compression, step by step

```
Top-K raw chunks (long) -> LLM reads each chunk + question -> keeps only relevant lines -> compressed context (short)
```

A 400-word policy clause becomes the one sentence that actually answers the question.

> **INTERVIEW Q:** "Why not just retrieve fewer chunks instead of compressing?" →
> Fewer chunks risks missing the answer entirely; compression keeps the recall but removes the
> noise *after* the fact.


In [ ]:
# Contextual Compression: retrieve as usual, then trim every chunk to only what answers the question.
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

question = "What is the grace period for premium payment?"
raw_docs = base_retriever.invoke(question)
compressed_docs = compression_retriever.invoke(question)

print("RAW chunk length:       ", len(raw_docs[0].page_content), "chars")
print("RAW chunk preview:      ", raw_docs[0].page_content[:300].replace("\n", " "))
print()
if compressed_docs:
    print("COMPRESSED chunk length:", len(compressed_docs[0].page_content), "chars")
    print("COMPRESSED chunk:       ", compressed_docs[0].page_content)
else:
    print("COMPRESSED: this chunk had nothing relevant and was dropped entirely.")


### Combine both fixes

In most real systems, multi-query and compression are stacked: multi-query widens the net at
retrieval time, compression cleans up whatever the net pulled in before it reaches the LLM.


In [ ]:
# Stack Multi-Query + Contextual Compression into one retriever.
optimized_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=multi_query_retriever,
)

optimized_result = answer_question(casual_question, optimized_retriever)
print("Question:", casual_question)
print("Answer:  ", optimized_result["answer"])


## Part 2 — Measuring What "Good" Means

You can't fix what you can't measure — an answer can *feel* right and still be wrong.

> **Why this matters:** in an interview or on the job, "it looks fine to me" is not an evaluation
> strategy. Real metrics are what separate a toy project from something a team can trust.

### Why evaluation actually matters
1. **Scale** — you can't personally read ten thousand answers before every release.
2. **Regression** — did last night's prompt tweak help or quietly hurt? Only a metric tells you.
3. **Trust** — stakeholders want numbers before they'll let you ship to real users.

> **REMEMBER THIS:** Evaluation isn't extra work on top of the project — it IS the project, once
> you're past the demo stage.


### Faithfulness vs. Relevance

The two questions every RAG answer needs to survive:

- **Faithfulness** — is the answer actually supported by the retrieved context, or did the model
  add something that isn't there? *(Faithfulness = no hallucination.)*
- **Relevance** — does the answer actually address what was asked, even if every word in it
  happens to be true? *(Relevance = actually answers the question.)*

> **WRITE THIS DOWN:** Classic example: ask "what is the capital of France?" and get back
> "Paris has a population of about 2 million." True, faithful, and completely irrelevant.

A system can fail either one independently:
- High faithfulness + low relevance = correct facts, wrong question answered.
- Low faithfulness + high relevance = sounds exactly like what was asked, but made up the details.

You need **both** scores to know what's actually going on.


In [ ]:
# Faithfulness scorer: is the answer grounded in the retrieved context?  (LLM-as-judge, single answer)
import json

JUDGE_MODEL = "gpt-4o-mini"
judge_llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0)


def _ask_judge_for_json(prompt: str) -> dict:
    response = judge_llm.invoke([HumanMessage(content=prompt)])
    text = response.content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"score": 0.0, "reason": f"Could not parse judge output: {text[:200]}"}


def faithfulness_score(question: str, context: str, answer: str) -> dict:
    """Score 0-1: is `answer` fully supported by `context`? 1 = no hallucination."""
    prompt = f"""You are grading FAITHFULNESS: is the ANSWER fully supported by the CONTEXT,
with no invented facts?

Return ONLY JSON: {{"score": <float 0-1>, "reason": "<one sentence>"}}
- 1.0 = every claim in the answer is backed by the context
- 0.5 = partially supported, some unsupported claims
- 0.0 = the answer contradicts or invents facts not in the context

CONTEXT:
{context}

ANSWER:
{answer}
"""
    return _ask_judge_for_json(prompt)


def relevance_score(question: str, answer: str) -> dict:
    """Score 0-1: does `answer` actually address `question`? 1 = directly on-topic."""
    prompt = f"""You are grading RELEVANCE: does the ANSWER actually address the QUESTION that
was asked, regardless of whether it is factually true?

Return ONLY JSON: {{"score": <float 0-1>, "reason": "<one sentence>"}}
- 1.0 = directly answers exactly what was asked
- 0.5 = partially on-topic, misses part of the question
- 0.0 = does not address the question at all

QUESTION:
{question}

ANSWER:
{answer}
"""
    return _ask_judge_for_json(prompt)


def combined_score(question: str, context: str, answer: str) -> dict:
    f = faithfulness_score(question, context, answer)
    r = relevance_score(question, answer)
    combined = round((f["score"] + r["score"]) / 2, 3)
    return {"faithfulness": f["score"], "faithfulness_reason": f["reason"],
            "relevance": r["score"], "relevance_reason": r["reason"], "combined": combined}


In [ ]:
# The "Paris" example from the slides -- faithful AND irrelevant.
paris_scores = combined_score(
    question="What is the capital of France?",
    context="Paris is the capital and most populous city of France.",
    answer="Paris has a population of about 2.1 million people.",
)
print(json.dumps(paris_scores, indent=2))


In [ ]:
# Run faithfulness + relevance over a small evaluation set from the insurance PDFs.
evaluation_set = [
    "What is the grace period for premium payment?",
    "What is the free look period in the group term policy?",
    "What is the maturity benefit in the Sanchay Plus policy?",
    "What happens if premium is not paid in the Sanchay Plus policy?",
    "List the exclusions mentioned in the policy.",
]

eval_results = []
for q in evaluation_set:
    result = answer_question(q, optimized_retriever)
    scores = combined_score(q, result["context"], result["answer"])
    eval_results.append({"question": q, "answer": result["answer"], **scores})
    print("=" * 70)
    print("Q:", q)
    print("A:", result["answer"][:200].replace("\n", " "))
    print(f"Faithfulness={scores['faithfulness']}  Relevance={scores['relevance']}  Combined={scores['combined']}")

avg_faithfulness = sum(r["faithfulness"] for r in eval_results) / len(eval_results)
avg_relevance = sum(r["relevance"] for r in eval_results) / len(eval_results)
print("\nAverage faithfulness:", round(avg_faithfulness, 3))
print("Average relevance:   ", round(avg_relevance, 3))


> **REMEMBER THIS:** A high faithfulness score with a low relevance score usually means one thing:
> correct answer, wrong question. Look at both numbers, never just one.


### You Don't Have to Hand-Roll This: RAGAS

`faithfulness_score` / `relevance_score` above are hand-rolled LLM-as-judge prompts -- great for
seeing exactly what "grading an answer" means under the hood, but in practice most teams reach
for a maintained library instead of hand-tuning their own judge prompts forever.
[RAGAS](https://github.com/explodinggraphs/ragas) is the standard one -- it implements
`Faithfulness` and `ResponseRelevancy` as ready-made, well-tested metrics using the exact same
idea (an LLM grading groundedness and on-topic-ness), plus a proper embedding-based relevancy
check instead of a single yes/no prompt.

```
pip install ragas
```

> **WRITE THIS DOWN:** This is the RAGAS pointer from the wrap-up slide, made concrete. If an
> interviewer asks "how would you evaluate a RAG system," "faithfulness and relevance, either
> hand-rolled or with RAGAS" is a complete, credible answer.


In [ ]:
# RAGAS: the library version of the exact two metrics we hand-rolled above.
from ragas import SingleTurnSample
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o-mini', temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))

ragas_faithfulness = Faithfulness(llm=ragas_llm)
ragas_relevancy = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

ragas_results = []
for q in evaluation_set:
    result = answer_question(q, optimized_retriever)
    sample = SingleTurnSample(
        user_input=q,
        retrieved_contexts=[d.page_content for d in result['docs']],
        response=result['answer'],
    )
    ragas_results.append({
        'question': q,
        'ragas_faithfulness': ragas_faithfulness.single_turn_score(sample),
        'ragas_relevancy': ragas_relevancy.single_turn_score(sample),
    })
    print('Q:', q)
    print(f"  RAGAS faithfulness={ragas_results[-1]['ragas_faithfulness']:.3f}"
          f"  RAGAS relevancy={ragas_results[-1]['ragas_relevancy']:.3f}")

avg_ragas_faith = sum(r['ragas_faithfulness'] for r in ragas_results) / len(ragas_results)
avg_ragas_rel = sum(r['ragas_relevancy'] for r in ragas_results) / len(ragas_results)
print('\nAverage RAGAS faithfulness:', round(avg_ragas_faith, 3))
print('Average RAGAS relevancy:   ', round(avg_ragas_rel, 3))
print('\n(Compare to the hand-rolled averages above -- expect close but not identical numbers:')
print(' different prompts, different grading logic, same underlying question.)')


### LLM-as-a-Judge

1. **What it is** — a strong LLM grades another model's answer against a rubric, like a teacher
   marking an exam.
2. **Why use it** — far cheaper and faster than a human reviewing every answer; scales to
   thousands of cases.
3. **Watch out for** — judges can favor longer, more confident-sounding answers. Spot-check
   against real humans.

> **INTERVIEW Q:** "How do you evaluate a RAG system when you don't have ground-truth labels?" →
> LLM-as-a-judge with a clear rubric is a standard, defensible answer.

> **WRITE THIS DOWN:** Direct, pairwise comparisons ("which answer is better") are often more
> reliable than asking a judge for an absolute 1-5 score.


In [ ]:
# Pairwise LLM-as-a-Judge: which of two candidate answers is better?
def pairwise_judge(question: str, answer_a: str, answer_b: str) -> dict:
    prompt = f"""You are comparing two candidate answers to the same question.
Judge which one is more helpful and direct for a real user.

Return ONLY JSON: {{"winner": "A" or "B", "reason": "<one sentence>"}}

QUESTION:
{question}

ANSWER A:
{answer_a}

ANSWER B:
{answer_b}
"""
    return _ask_judge_for_json(prompt)


# The password-reset example from the slides.
password_question = "How do I reset my password?"
answer_a = "Click Forgot Password on the login page, check your email for a link, and set a new password."
answer_b = "Passwords can be reset through the account settings menu if you are logged in, or via support if not."

verdict = pairwise_judge(password_question, answer_a, answer_b)
print(json.dumps(verdict, indent=2))


In [ ]:
# Same idea on our own domain: compare the BASELINE retriever's answer vs the OPTIMIZED retriever's answer.
insurance_question = "What happens if I'm late on my payment?"

baseline_answer = answer_question(insurance_question, base_retriever)["answer"]
optimized_answer = answer_question(insurance_question, optimized_retriever)["answer"]

verdict = pairwise_judge(insurance_question, baseline_answer, optimized_answer)
print("Baseline answer: ", baseline_answer[:200].replace("\n", " "))
print("Optimized answer:", optimized_answer[:200].replace("\n", " "))
print()
print("Judge verdict (A=baseline, B=optimized):", json.dumps(verdict, indent=2))


## Part 3 — Seeing Inside the Black Box (LangSmith)

Your chain runs in milliseconds. When it's wrong, you need to see exactly which step failed.

> **Why this matters:** "it's not working" is not a bug report. A trace that shows the exact
> prompt, the retrieved chunks, and the raw model output is.

### What is LLM observability?
1. **Trace every call** — see the exact prompt, retrieved chunks, and output for every request.
2. **Debug fast** — find the failing step in seconds instead of guessing and re-running blind.
3. **Prove quality over time** — track scores and latency across versions, not just at launch.

> **REMEMBER THIS:** Observability is what evaluation looks like once your bot is live and
> talking to real users, not just your test set.

### LangSmith's core concepts

`Project → Trace → Run`

- **Project** = your whole app (e.g. "Insurance Support Bot").
- **Trace** = one full user request, start to finish.
- **Run** = one step inside that trace — a retrieval call, an LLM call, a tool call.


In [ ]:
# Connect a LangSmith Client so we can inspect what we've already traced.
from langsmith import Client

ls_client = Client()
project_name = os.environ["LANGCHAIN_PROJECT"]

print("Every cell above already sent traces to LangSmith because LANGSMITH_TRACING=true.")
print(f"Open https://smith.langchain.com and look at project: {project_name!r}")

recent_runs = list(ls_client.list_runs(project_name=project_name, limit=5))
print(f"\nMost recent {len(recent_runs)} runs in this project:")
for r in recent_runs:
    print("-", r.name, "|", r.run_type, "|", r.status)


### Two ways to get traces

- **Automatic (LangChain)** — if you're already using LangChain, tracing is often just one
  environment variable away (`LANGSMITH_TRACING=true`, done above). Every chain and retriever
  call gets captured for free.
- **Manual (`@traceable`)** — for your own custom Python functions, outside LangChain, wrap them
  with the `@traceable` decorator to log inputs, outputs, and timing.

> **REMEMBER THIS:** You don't have to choose one — most real projects mix both: automatic
> tracing for the LangChain parts, `@traceable` for the custom glue code around it.


In [ ]:
# Manual tracing with @traceable -- for the custom (non-LangChain) glue code around your pipeline.
from langsmith import traceable

@traceable(name="faithfulness_score", run_type="llm")
def traced_faithfulness_score(question: str, context: str, answer: str) -> dict:
    return faithfulness_score(question, context, answer)


@traceable(name="rag_pipeline", run_type="chain")
def traced_rag_pipeline(question: str) -> dict:
    """Custom glue function: optimized retrieval -> answer -> faithfulness/relevance scoring.
    LangChain calls inside here trace automatically; this decorator adds the OUTER step too."""
    result = answer_question(question, optimized_retriever)
    scores = combined_score(question, result["context"], result["answer"])
    return {**result, **scores}


traced_output = traced_rag_pipeline("What is the free look period in the group term policy?")
print("Answer:    ", traced_output["answer"][:200].replace("\n", " "))
print("Faithfulness:", traced_output["faithfulness"], "| Relevance:", traced_output["relevance"])
print("\nCheck LangSmith: you should see a 'rag_pipeline' trace containing nested retriever/LLM runs.")


### Datasets, evaluators & feedback

1. **Datasets** — a saved set of test questions you re-run every time you change something.
2. **Custom evaluators** — your own faithfulness/relevance scoring functions, run automatically
   on every trace.
3. **Feedback scoring** — attach a thumbs-up/down or a numeric score directly onto a specific run.

> **REMEMBER THIS:** A dataset plus an evaluator turns "did I break anything" from a guess into a
> one-click re-run before every deploy.


In [ ]:
# LangSmith Dataset: save evaluation_set once, re-run it every time the pipeline changes.
dataset_name = "week6-insurance-eval-set"

try:
    dataset = ls_client.create_dataset(dataset_name=dataset_name,
                                        description="Insurance policy Q&A used to score RAG changes.")
    for q in evaluation_set:
        ls_client.create_example(inputs={"question": q}, dataset_id=dataset.id)
    print(f"Created dataset {dataset_name!r} with {len(evaluation_set)} examples.")
except Exception as e:
    # Dataset with this name already exists from a previous run -- reuse it.
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"Reusing existing dataset {dataset_name!r} ({e.__class__.__name__}).")


In [ ]:
# Custom evaluators wired to our faithfulness/relevance judges, run automatically over the dataset.
from langsmith.evaluation import evaluate
from langsmith.schemas import Run, Example


def target(inputs: dict) -> dict:
    """The 'system under test': our optimized RAG pipeline."""
    result = answer_question(inputs["question"], optimized_retriever)
    return {"answer": result["answer"], "context": result["context"]}


def faithfulness_evaluator(run: Run, example: Example) -> dict:
    outputs = run.outputs or {}
    score = faithfulness_score(example.inputs["question"], outputs.get("context", ""), outputs.get("answer", ""))
    return {"key": "faithfulness", "score": score["score"], "comment": score["reason"]}


def relevance_evaluator(run: Run, example: Example) -> dict:
    outputs = run.outputs or {}
    score = relevance_score(example.inputs["question"], outputs.get("answer", ""))
    return {"key": "relevance", "score": score["score"], "comment": score["reason"]}


experiment_results = evaluate(
    target,
    data=dataset_name,
    evaluators=[faithfulness_evaluator, relevance_evaluator],
    experiment_prefix="optimized-retriever",
)
print("Experiment submitted. Open the dataset in LangSmith to see per-example scores.")


In [ ]:
# Feedback scoring: attach a human thumbs-up/down directly onto a specific run.
recent_runs = list(ls_client.list_runs(project_name=project_name, limit=1))

if recent_runs:
    target_run = recent_runs[0]
    ls_client.create_feedback(
        run_id=target_run.id,
        key="user_rating",
        score=1,                      # 1 = thumbs up, 0 = thumbs down
        comment="Looked correct and on-topic during the live session.",
    )
    print(f"Feedback attached to run: {target_run.name} ({target_run.id})")
else:
    print("No runs found yet -- run a cell above first so there's something to give feedback on.")


### Comparing runs & the Prompt Hub

- **Compare Runs** — put two versions of your pipeline side by side (same questions, different
  prompts or settings) and see exactly where scores moved. This is exactly what running the same
  `dataset_name` through `evaluate()` with a different `experiment_prefix` (e.g. `"baseline-retriever"`
  vs `"optimized-retriever"`) gives you inside the LangSmith UI.
- **Prompt Hub** — a shared library of reusable, versioned prompts, so your best prompt isn't
  buried in one notebook cell.

> **INTERVIEW Q:** "How do you know a prompt change actually improved things?" → Run both versions
> against the same dataset in LangSmith and compare the scores side by side, don't just eyeball a
> few examples.


In [ ]:
# Run the BASELINE retriever through the same dataset -- so LangSmith can compare it against
# the "optimized-retriever" experiment above, side by side, on identical questions.
def baseline_target(inputs: dict) -> dict:
    result = answer_question(inputs["question"], base_retriever)
    return {"answer": result["answer"], "context": result["context"]}


baseline_experiment_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=[faithfulness_evaluator, relevance_evaluator],
    experiment_prefix="baseline-retriever",
)
print("Baseline experiment submitted.")
print("In LangSmith, open the dataset -> 'Compare' -> pick 'baseline-retriever' vs 'optimized-retriever'.")


## Part 4 — LangSmith vs. Langfuse

Same job — watch every LLM call — but two very different companies behind the tool.

> **Why this matters:** you will get asked "which observability tool have you used" in interviews.
> Knowing both, and why you'd pick one over the other, is the actual signal.

### What is Langfuse?
1. **Open-source core** — the core tracing and eval engine is open-source; read the code,
   self-host it, or use their cloud.
2. **Framework-agnostic** — works with LangChain, LlamaIndex, or plain API calls.
3. **Tracing + evals + prompts** — covers the same ground as LangSmith: traces, datasets,
   evaluators, and prompt management.

> **REMEMBER THIS:** Langfuse being open-source doesn't just mean it's free — it means a company
> can run it entirely inside their own infrastructure, which matters a lot for regulated
> industries like insurance or healthcare.

### Head to head

| LangSmith | Langfuse |
|---|---|
| Built by the LangChain team | Open-source core, self-host or cloud |
| Deepest native fit for LangChain / LangGraph | Framework-agnostic — LangChain, LlamaIndex, raw API |
| Managed SaaS, with a free tier | Free if self-hosted; usage-based cloud tier |
| Strong Prompt Hub for shared prompts | Strong on cost & latency dashboards |
| Pricing scales with trace volume | Growing eval + prompt management features |

> **REMEMBER THIS:** Rule of thumb — all-in on LangChain and want the path of least resistance →
> LangSmith. Need self-hosting, data residency, or you're framework-agnostic → Langfuse.


In [ ]:
# Langfuse pattern (OPTIONAL - runs only if `langfuse` is installed AND keys are set in .env).
# `pip install langfuse` and add LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_HOST to try it live.
try:
    from langfuse import observe, get_client
    LANGFUSE_INSTALLED = True
except ImportError:
    LANGFUSE_INSTALLED = False

langfuse_configured = bool(os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY"))

if LANGFUSE_INSTALLED and langfuse_configured:
    langfuse_client = get_client()

    @observe(name="rag_pipeline_langfuse")
    def langfuse_traced_pipeline(question: str) -> dict:
        result = answer_question(question, optimized_retriever)
        scores = combined_score(question, result["context"], result["answer"])
        langfuse_client.score_current_trace(name="faithfulness", value=scores["faithfulness"])
        langfuse_client.score_current_trace(name="relevance", value=scores["relevance"])
        return result

    output = langfuse_traced_pipeline("What is the maturity benefit in the Sanchay Plus policy?")
    print("Traced to Langfuse. Answer:", output["answer"][:200])
else:
    print("Langfuse not configured -- this cell is intentionally skipped.")
    print("The pattern to remember: same idea as @traceable, but the decorator is @observe,")
    print("and instead of `ls_client.create_feedback(...)` you call `langfuse_client.score_current_trace(...)`.")


## Part 5 — Hands-On: Build It Yourself

Reading about multi-query retrieval and watching it fix a bad answer are not the same thing.
Type the code yourself, break it on purpose at least once, and look at what the trace shows you
when something fails.

### Exercise 1 — Multi-query on your own tricky phrasing
Pick a casual, real-user-style question about one of the 3 PDFs. Compare `base_retriever` vs
`multi_query_retriever` on it. Does multi-query pull in the chunk that baseline missed?

### Exercise 2 — Measure the compression savings
For 3 questions, compare `len(raw context)` vs `len(compressed context)`. What's the average
percentage reduction? Does the compressed answer's faithfulness score go up, down, or stay flat?

### Exercise 3 — Prove the optimization actually helped
Run `combined_score(...)` for the same 5 questions through `base_retriever` AND
`optimized_retriever`. Compare the average faithfulness and relevance. This is the number you'd
show a stakeholder.

### Exercise 4 — Break it, trace it, fix it
Ask a question you're pretty sure the bot will get wrong (something not in the 3 PDFs, or phrased
very unusually). Open the trace in LangSmith, find the exact step that failed, and fix it using
one of today's techniques.


In [ ]:
# Exercise 1 starter -- fill in your own question below.
my_question = "how much extra cover can I add on top?"   # try your own phrasing here

baseline_hit = base_retriever.invoke(my_question)
multiquery_hit = multi_query_retriever.invoke(my_question)

print("Baseline top chunk:  ", baseline_hit[0].page_content[:150].replace("\n", " "))
print("Multi-query # chunks:", len(multiquery_hit))


In [ ]:
# Exercise 3 starter -- baseline vs optimized, averaged over the evaluation set.
def average_scores(retriever, questions):
    scores = [combined_score(q, answer_question(q, retriever)["context"], answer_question(q, retriever)["answer"])
              for q in questions]
    return {
        "avg_faithfulness": round(sum(s["faithfulness"] for s in scores) / len(scores), 3),
        "avg_relevance": round(sum(s["relevance"] for s in scores) / len(scores), 3),
    }

print("Baseline: ", average_scores(base_retriever, evaluation_set))
print("Optimized:", average_scores(optimized_retriever, evaluation_set))


## Wrap-Up — Honest Limitations & What's Next

### Honest limitations
- LLM judges can disagree with human reviewers — spot-check regularly.
- Judges and tracing both cost extra tokens and latency.
- More tools (LangSmith + Langfuse) means more setup and upkeep.
- None of this fixes a genuinely bad or missing document.

> **REMEMBER THIS:** RAG is powerful but not magic — its answer quality is only as good as the
> data, the retrieval, and the evaluation behind it.

### Where you go next
1. **RAGAS framework** — you already ran it above (`Faithfulness`, `ResponseRelevancy`); the
   library covers more metrics built on the same ideas as today (context precision/recall, answer
   correctness against a gold reference, and more).
2. **Cost-aware evaluation** — tracking answer quality per rupee/dollar spent, not quality alone.
3. **Production monitoring** — live alerts when faithfulness or relevance drops on real traffic.

### Recap in one breath
We made retrieval smarter with multi-query and compression, we learned to actually measure
answers with faithfulness, relevance, and LLM-as-a-judge, and we opened up the black box with
LangSmith while getting comfortable naming Langfuse as a real alternative.

**Homework mindset:** don't just re-run this notebook once and move on. Take one of your own
questions against the insurance PDFs, deliberately try to break your bot with it, then use
tracing to figure out exactly why it broke, and fix it using one of today's techniques.
That loop — break it, trace it, fix it — is basically the day-to-day job of anyone working on
production LLM systems.
